In [1]:
import torch
import torch.nn as nn

from torchrl.envs.utils import step_mdp
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.data.replay_buffers import ReplayBuffer, LazyTensorStorage
from torchrl.modules import SafeModule, ProbabilisticActor, ValueOperator
from torchrl.modules.distributions import NormalParamExtractor, TanhNormal
from torchrl.objectives.sac import SACLoss
from torchrl.objectives import SoftUpdate, group_optimizers

from ksp_state import KSPState
from qvn_model import QVNModel

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

Using device: cuda


For this RL problem I will be using the SAC (Soft Actor Critic) algorithm. This is an algorithm that finds the best stochastic policy: pi, and the best Q(s,a) function. We do this by approximating their values though two neural networks: theta and phi. So we estimate pi(.|s ; phi) and Q(s,a | theta). We then use a distribution to choose our actions. The second crucial part of the algorithm is that the objective function: J(theta) includes a "entropy term" which is maximized. Meaning that the model is allowed to explore the space that it is given.

Kerbal Space Program is a continious problem, I thought about making this a sequential problem by allowing "frames" where the agent would act (say every 4 frames), as in the original DQN paper. Yet Kerbal space program needs inputs that are continious such as thrust (eg sometime I need the rockets thrust to be 10% to land and not intermitent from 100% to 10%). So I discarded DQC, DDQC, and other competing algorithms for SAC. Note that the implementation of SAC that I will be using (from torchRL) was chosen due to already implementing hyperparameter optimization and two competing Q-networks, which we choose the minimum value of to minimize training time.

At saying this our primary job is to implement an enviorment using KRPC that is compatible with torchRL's enviorments. This way in the future, and maybe even within the same project we could get to the point where we can implement and compare many algorithms. A modified enviorment might still allow DDQN, PPO and more.

DAC came from both robotics, as it is generally the most popular algorithm in the area but also a series of youtube videos about training an RL agent to beat world record in the racing game: trackmania. There was an offhand mention of the algorithm in one of the videos so I decided to investigate and came upon the original DAC paper and code in github.

In [2]:
env = KSPState(
    target_orbit_altitude=80_000,
    step_interval=0.5,
    max_steps=2000,
)

N_OBS = 10
N_ACTIONS = 3
HIDDEN = 256



In [3]:
actor_net = nn.Sequential(
    nn.Linear(N_OBS, HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, 2 * N_ACTIONS),
    NormalParamExtractor(),
)

actor_module = SafeModule(
    actor_net,
    in_keys=["observation"],
    out_keys=["loc", "scale"],
)
 
actor = ProbabilisticActor(
    module=actor_module,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    spec=env.action_spec,
    # TanhNormal keeps actions inside the environment bounds.
    distribution_class=TanhNormal,
).to(device)
 


In [4]:
qvalue = ValueOperator(
    module=QVNModel(N_OBS, N_ACTIONS, HIDDEN),
    in_keys=["observation", "action"],
).to(device)

Here we can leverage the compatibility of the state created with the torchRL package. Thus we can use the built in SAC algorithm, and specifically use the two q-networks as noted by 1801.01290v2

In [5]:
loss_module = SACLoss(
    actor_network=actor,
    qvalue_network=qvalue,
    num_qvalue_nets=2,
    alpha_init=0.2,
    target_entropy="auto",
    loss_function="smooth_l1",
)
loss_module.to(device)

# Soft target updates keep the critic targets from moving too abruptly.
target_updater = SoftUpdate(loss_module, tau=0.005)


optimizer_actor = torch.optim.Adam(
    loss_module.actor_network_params.values(True, True), lr=3e-4
)
optimizer_critic = torch.optim.Adam(
    loss_module.qvalue_network_params.values(True, True), lr=3e-4
)
optimizer_alpha = torch.optim.Adam(
    [loss_module.log_alpha], lr=3e-4
)
optimizer = group_optimizers(optimizer_actor, optimizer_critic, optimizer_alpha)
del optimizer_actor, optimizer_critic, optimizer_alpha

In [6]:
import pickle
from pathlib import Path
from tensordict import TensorDict

USER_DATA_DIR = Path("userData")
USER_DATA_DIR.mkdir(exist_ok=True)
DEFAULT_DEMO_PATH = USER_DATA_DIR / "demo_buffer.pkl"

def load_demos_into_buffer(buffer, filename=DEFAULT_DEMO_PATH, n_copies=5):
    # Duplicate demos so successful flights are sampled more often early on.
    filename = Path(filename)
    with open(filename, "rb") as f:
        transitions = pickle.load(f)

    for _ in range(n_copies):
        for t in transitions:
            td = TensorDict({
                "observation": t["observation"],
                "action": t["action"],
                "next": TensorDict({
                    "observation": t["next_observation"],
                    "reward": t["reward"],
                    "done": t["done"],
                    "terminated": t["terminated"],
                    "truncated": t["truncated"],
                }, batch_size=[]),
            }, batch_size=[])
            buffer.add(td)

    print(f"Loaded {len(transitions)} transitions x {n_copies} copies "
          f"= {len(transitions) * n_copies} total into buffer")

The following code records demos, this was because I found that giving the AI one or two demonstrations of a successfull orbit would increase the convergance of the models significantly.

In [16]:

import time
import torch
import pickle
from pathlib import Path
from ksp_state import KSPState

def record_demo(filename=DEFAULT_DEMO_PATH, step_interval=0.5):
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)
    env = KSPState(target_orbit_altitude=80_000, step_interval=step_interval)
    td = env.reset()

    transitions = []
    print("Recording — fly the craft manually in KSP!")
    print("Press Ctrl+C to stop recording.\n")

    try:
        step = 0
        while True:

            pitch = env.vessel.flight(env.body.reference_frame).pitch
            heading = env.vessel.flight(env.body.reference_frame).heading

            prev_obs = td["observation"].clone()

            # Sample manual input on the same cadence used during training.
            t_start = env.space_center.ut
            while env.space_center.ut < t_start + step_interval:
                time.sleep(0.01)


            new_obs = env._get_obs()
            intact = env._vessel_intact()


            new_throttle = env.vessel.control.throttle
            new_pitch = env.vessel.flight(env.body.reference_frame).pitch
            new_heading = env.vessel.flight(env.body.reference_frame).heading

            action = torch.tensor([
                new_throttle * 2.0 - 1.0,
                (new_pitch - pitch) / 5.0,
                ((new_heading - heading + 180) % 360 - 180) / 5.0
            ], dtype=torch.float32).clamp(-1.0, 1.0)

            reward = env._reward_function(prev_obs, new_obs, intact)

            transitions.append({
                "observation": prev_obs,
                "action": action,
                "next_observation": new_obs,
                "reward": torch.tensor([reward], dtype=torch.float32),
                "done": torch.tensor([not intact]),
                "terminated": torch.tensor([not intact]),
                "truncated": torch.tensor([False]),
            })

            td["observation"] = new_obs
            env._prev_obs = new_obs
            step += 1

            if step % 20 == 0:
                apo = new_obs[1].item()
                peri = new_obs[2].item()
                alt = new_obs[0].item()
                print(f"Step {step:4d} | alt: {alt:.3f} | apo: {apo:.3f} | peri: {peri:.3f}")

            if not intact:
                print("Vessel destroyed.")
                break

            if new_obs[2].item() >= 1.0 and new_obs[1].item() >= 1.0:
                print("ORBIT ACHIEVED!")
                break

    except KeyboardInterrupt:
        print("\nRecording stopped.")

    with open(filename, "wb") as f:
        pickle.dump(transitions, f)

    print(f"Saved {len(transitions)} transitions to {filename}")
    env.close()


if __name__ == "__main__":
    record_demo()

Recording — fly the craft manually in KSP!
Press Ctrl+C to stop recording.

Step   20 | alt: 0.007 | apo: 0.012 | peri: -7.480
Step   40 | alt: 0.026 | apo: 0.056 | peri: -7.480
Step   60 | alt: 0.058 | apo: 0.105 | peri: -7.480
Step   80 | alt: 0.097 | apo: 0.166 | peri: -7.480
Step  100 | alt: 0.143 | apo: 0.247 | peri: -7.480
Step  120 | alt: 0.199 | apo: 0.355 | peri: -7.481
Step  140 | alt: 0.267 | apo: 0.501 | peri: -7.481
Step  160 | alt: 0.348 | apo: 0.659 | peri: -7.449
Step  180 | alt: 0.433 | apo: 0.765 | peri: -7.337
Step  200 | alt: 0.519 | apo: 0.882 | peri: -7.134
Step  220 | alt: 0.601 | apo: 0.940 | peri: -6.769
Step  240 | alt: 0.679 | apo: 1.042 | peri: -6.340
Step  260 | alt: 0.755 | apo: 1.133 | peri: -5.946
Step  280 | alt: 0.825 | apo: 1.133 | peri: -5.947
Step  300 | alt: 0.888 | apo: 1.133 | peri: -5.947
Step  320 | alt: 0.944 | apo: 1.133 | peri: -5.947
Step  340 | alt: 0.995 | apo: 1.133 | peri: -5.947
Step  360 | alt: 1.037 | apo: 1.133 | peri: -5.947
Step  

In [7]:
buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=100_000),
    batch_size=256,
)

In [15]:
import hashlib
import io
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def _checkpoint_episode(path):
    name = Path(path).stem
    if name.startswith("checkpoint_ep"):
        suffix = name.removeprefix("checkpoint_ep")
        return int(suffix) if suffix.isdigit() else None
    return None


def list_training_checkpoints(checkpoint_dir=DATA_DIR):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_paths = sorted(
        checkpoint_dir.glob("checkpoint_ep*.pt"),
        key=lambda path: _checkpoint_episode(path) or -1,
    )

    latest_path = checkpoint_dir / "checkpoint_latest.pt"
    if latest_path.exists():
        checkpoint_paths.append(latest_path)

    return checkpoint_paths


def _transition_fingerprint(td):
    cpu_td = td.detach().cpu()
    payload = io.BytesIO()
    torch.save(cpu_td, payload)
    return hashlib.sha1(payload.getvalue()).hexdigest()


def save_training(filepath, episode, total_steps, loss_module, optimizer, buffer):
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    buffer_snapshot = buffer[:]
    if hasattr(buffer_snapshot, "cpu"):
        buffer_snapshot = buffer_snapshot.cpu()

    torch.save({
        'episode': episode,
        'total_steps': total_steps,
        'loss_module': loss_module.state_dict(),
        'optimizer': optimizer.state_dict(),
        'buffer': buffer_snapshot,
    }, filepath)
    print(f"Saved training state to {filepath}")


def load_training(filepath, loss_module, optimizer, map_location="cpu"):
    ckpt = torch.load(filepath, weights_only=False, map_location=map_location)
    incompatible = loss_module.load_state_dict(ckpt['loss_module'], strict=False)
    if incompatible.missing_keys:
        print(f"Missing checkpoint keys ignored: {incompatible.missing_keys}")
    if incompatible.unexpected_keys:
        print(f"Unexpected checkpoint keys ignored: {incompatible.unexpected_keys}")

    if 'optimizer' in ckpt:
        try:
            optimizer.load_state_dict(ckpt['optimizer'])
        except Exception as exc:
            print(f"Optimizer state could not be fully restored: {exc}")

    start_episode = ckpt['episode'] + 1
    total_steps = ckpt['total_steps']
    print(f"Resuming model state from {filepath} at episode {start_episode}, total_steps {total_steps}")
    return ckpt, start_episode, total_steps


def restore_training(filepath, loss_module, optimizer, buffer, map_location="cpu"):
    ckpt, start_episode, total_steps = load_training(
        filepath,
        loss_module,
        optimizer,
        map_location=map_location,
    )

    restored = 0
    saved_buffer = ckpt.get("buffer")
    if saved_buffer is not None:
        if hasattr(saved_buffer, "cpu"):
            saved_buffer = saved_buffer.cpu()
        buffer.extend(saved_buffer)
        restored = len(saved_buffer)

    print(f"Restored {restored} transitions in replay buffer from {filepath}")
    return start_episode, total_steps


def restore_training_from_checkpoints(loss_module, optimizer, buffer, checkpoint_dir=DATA_DIR, device="cpu"):
    checkpoint_paths = list_training_checkpoints(checkpoint_dir)
    if not checkpoint_paths:
        raise FileNotFoundError("No training checkpoints found")

    print("Found checkpoints:", ", ".join(path.name for path in checkpoint_paths))

    newest_checkpoint = checkpoint_paths[-1]
    _, start_episode, total_steps = load_training(
        newest_checkpoint,
        loss_module,
        optimizer,
        map_location=device,
    )

    seen = set()
    restored = 0
    for checkpoint_path in checkpoint_paths:
        ckpt = torch.load(checkpoint_path, weights_only=False, map_location="cpu")
        saved_buffer = ckpt.get("buffer")
        if saved_buffer is None:
            continue

        for idx in range(len(saved_buffer)):
            transition = saved_buffer[idx].detach().cpu()
            fingerprint = _transition_fingerprint(transition)
            if fingerprint in seen:
                continue

            seen.add(fingerprint)
            buffer.add(transition)
            restored += 1

    print(f"Restored {restored} unique transitions into replay buffer (buffer size now {len(buffer)})")
    return start_episode, total_steps

In [ ]:
env.close()

In [13]:
import csv
import math
import random
from pathlib import Path

BATCH_SIZE = 256
NUM_EPISODES = 500
WARMUP_STEPS = 1000
UPDATES_PER_STEP = 1
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
USER_DATA_DIR = Path("userData")
USER_DATA_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = DATA_DIR
CHECKPOINT_PATH = CHECKPOINT_DIR / "checkpoint_latest.pt"
DEMO_BUFFER_PATH = USER_DATA_DIR / "demo_buffer.pkl"
SEED = 0

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
env._set_seed(SEED)

LOG_DIR = DATA_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
STEP_LOG_PATH = LOG_DIR / "training_step_metrics.csv"
EPISODE_LOG_PATH = LOG_DIR / "training_episode_metrics.csv"

STEP_LOG_FIELDS = [
    "seed", "train_step", "episode", "step_in_episode", "eval_flag", "success",
    "orbit_time_s", "total_return", "reward_total", "reward_potential",
    "reward_alive", "reward_overshoot", "reward_orbit_bonus",
    "reward_failure_penalty", "reward_ground_penalty", "fuel_remaining",
    "fuel_remaining_frac", "apoapsis", "periapsis", "altitude",
    "max_altitude", "speed", "max_speed", "termination_reason",
    "actor_loss", "critic_loss",
    "alpha", "loss_alpha",
]

EPISODE_LOG_FIELDS = [
    "seed", "train_step", "episode", "eval_flag", "success", "orbit_time_s",
    "total_return", "fuel_remaining", "fuel_remaining_frac", "apoapsis",
    "periapsis", "max_altitude", "max_speed", "termination_reason",
    "episode_steps",
]

def _open_csv_writer(path, fieldnames):
    expected_header = ",".join(fieldnames)
    mode = "a"
    needs_header = True
    if path.exists() and path.stat().st_size > 0:
        with path.open("r", encoding="utf-8", newline="") as existing_file:
            current_header = existing_file.readline().strip()
        if current_header == expected_header:
            mode = "a"
            needs_header = False
        else:
            mode = "w"
            needs_header = True
    else:
        mode = "w"
        needs_header = True

    handle = path.open(mode, newline="", encoding="utf-8", buffering=1)
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    if needs_header:
        writer.writeheader()
        handle.flush()
    return handle, writer


def _as_float(value):
    if value is None:
        return math.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        return math.nan


# Resume the model and replay buffer if a checkpoint is available.
if CHECKPOINT_PATH.exists():
    start_episode, total_steps = restore_training(
        CHECKPOINT_PATH,
        loss_module,
        optimizer,
        buffer,
        map_location=device,
    )
else:
    start_episode = 0
    total_steps = 0

if DEMO_BUFFER_PATH.exists():
    load_demos_into_buffer(buffer, DEMO_BUFFER_PATH, n_copies=5)

step_log_handle, step_log_writer = _open_csv_writer(STEP_LOG_PATH, STEP_LOG_FIELDS)
episode_log_handle, episode_log_writer = _open_csv_writer(EPISODE_LOG_PATH, EPISODE_LOG_FIELDS)

try:
    for episode in range(start_episode, NUM_EPISODES):
        td = env.reset()
        episode_reward = 0.0
        episode_steps = 0

        while True:
            actor_loss = math.nan
            critic_loss = math.nan
            alpha_value = _as_float(loss_module._alpha.item())
            loss_alpha = math.nan

            if total_steps < WARMUP_STEPS:
                td["action"] = torch.rand(3) * 2.0 - 1.0
            else:
                td_device = td.to(device)
                with torch.no_grad(), set_exploration_type(ExplorationType.RANDOM):
                    td_device = actor(td_device)
                td["action"] = td_device["action"].cpu()
                td["loc"] = td_device["loc"].cpu()
                td["scale"] = td_device["scale"].cpu()

            next_td = env.step(td)
            step_info = env.get_step_info()

            reward_value = next_td["next", "reward"].item()
            episode_reward += reward_value
            episode_steps += 1
            total_steps += 1

            # Store every transition so the replay buffer tracks the full run.
            buffer.add(next_td)

            if total_steps >= WARMUP_STEPS and len(buffer) >= BATCH_SIZE:
                for _ in range(UPDATES_PER_STEP):
                    batch = buffer.sample().to(device)
                    loss_td = loss_module(batch)

                    total_loss = (
                        loss_td["loss_actor"]
                        + loss_td["loss_qvalue"]
                        + loss_td["loss_alpha"]
                    )

                    optimizer.zero_grad(set_to_none=True)
                    total_loss.backward()
                    optimizer.step()
                    target_updater.step()

                    actor_loss = _as_float(loss_td["loss_actor"].item())
                    critic_loss = _as_float(loss_td["loss_qvalue"].item())
                    alpha_value = _as_float(loss_module._alpha.item())
                    loss_alpha = _as_float(loss_td["loss_alpha"].item())

            # Keep enough per-step state for plots and summary tables later.
            step_row = {
                "seed": SEED,
                "train_step": total_steps,
                "episode": episode,
                "step_in_episode": step_info.get("step_in_episode", episode_steps),
                "eval_flag": False,
                "success": bool(step_info.get("success", False)),
                "orbit_time_s": _as_float(step_info.get("orbit_time_s")),
                "total_return": episode_reward,
                "reward_total": _as_float(step_info.get("reward_total", reward_value)),
                "reward_potential": _as_float(step_info.get("reward_potential")),
                "reward_alive": _as_float(step_info.get("reward_alive")),
                "reward_overshoot": _as_float(step_info.get("reward_overshoot")),
                "reward_orbit_bonus": _as_float(step_info.get("reward_orbit_bonus")),
                "reward_failure_penalty": _as_float(step_info.get("reward_failure_penalty")),
                "reward_ground_penalty": _as_float(step_info.get("reward_ground_penalty")),
                "fuel_remaining": _as_float(step_info.get("fuel_remaining")),
                "fuel_remaining_frac": _as_float(step_info.get("fuel_remaining_frac")),
                "apoapsis": _as_float(step_info.get("apoapsis")),
                "periapsis": _as_float(step_info.get("periapsis")),
                "altitude": _as_float(step_info.get("altitude")),
                "max_altitude": _as_float(step_info.get("max_altitude")),
                "speed": _as_float(step_info.get("speed")),
                "max_speed": _as_float(step_info.get("max_speed")),
                "termination_reason": step_info.get("termination_reason", ""),
                "actor_loss": actor_loss,
                "critic_loss": critic_loss,
                "alpha": alpha_value,
                "loss_alpha": loss_alpha,
            }
            step_log_writer.writerow(step_row)
            step_log_handle.flush()

            if next_td["next", "done"].item():
                final_info = dict(step_info)
                break

            td = step_mdp(next_td)

        episode_row = {
            "seed": SEED,
            "train_step": total_steps,
            "episode": episode,
            "eval_flag": False,
            "success": bool(final_info.get("success", False)),
            "orbit_time_s": _as_float(final_info.get("orbit_time_s")),
            "total_return": episode_reward,
            "fuel_remaining": _as_float(final_info.get("fuel_remaining")),
            "fuel_remaining_frac": _as_float(final_info.get("fuel_remaining_frac")),
            "apoapsis": _as_float(final_info.get("apoapsis")),
            "periapsis": _as_float(final_info.get("periapsis")),
            "max_altitude": _as_float(final_info.get("max_altitude")),
            "max_speed": _as_float(final_info.get("max_speed")),
            "termination_reason": final_info.get("termination_reason", ""),
            "episode_steps": episode_steps,
        }
        episode_log_writer.writerow(episode_row)
        episode_log_handle.flush()

        if episode > 0 and episode % 50 == 0:
            save_training(
                CHECKPOINT_DIR / f"checkpoint_ep{episode}.pt",
                episode,
                total_steps,
                loss_module,
                optimizer,
                buffer,
            )

        save_training(
            CHECKPOINT_PATH,
            episode,
            total_steps,
            loss_module,
            optimizer,
            buffer,
        )

        print(
            f"Episode {episode:4d} | "
            f"steps: {episode_steps:5d} | "
            f"total: {total_steps:7d} | "
            f"reward: {episode_reward:8.2f} | "
            f"orbit: {'YES' if final_info.get('success', False) else 'no'} | "
            f"reason: {final_info.get('termination_reason', '')}"
        )
finally:
    step_log_handle.close()
    episode_log_handle.close()
    env.close()

2026-04-13 10:36:10,985 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]
Saved training state to data\checkpoint_latest.pt
Episode    0 | steps:    59 | total:      59 | reward:   -40.21 | orbit: no | reason: vessel_destroyed
Saved training state to data\checkpoint_latest.pt
Episode    1 | steps:   111 | total:     170 | reward:   -42.24 | orbit: no | reason: vessel_destroyed
Saved training state to data\checkpoint_latest.pt
Episode    2 | steps:   122 | total:     292 | reward:   -41.89 | orbit: no | reason: vessel_destroyed
Saved training state to data\checkpoint_latest.pt
Episode    3 | steps:    20 | total:     312 | reward:   -40.05 | orbit: no | reason: vessel_destroyed
Saved training state to data\checkpoint_latest.pt
Episode    4 | steps:    20 | total:     332 | reward:   -40.02 | orbit: no | reason: vessel_destroyed
Saved training state to data\checkpoint_latest.pt
Episode    5 | steps:    70 | total:     402 | reward:   -40.72 | orbit: n

KeyboardInterrupt: 

In [ ]:
env.close()